In [70]:
import pandas as pd
import torch
import os
import math
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import seaborn as sns
import matplotlib.pyplot as plt

In [47]:
representations = torch.load("merged.pt") # first vector of the rinalmo results for all RNAs
rfam_ids = pd.read_csv("sequence_rfam_mapping_annotations.csv") #their corresponding rfam ids (family)
alignment_score_matrix = pd.read_csv("alignment_matrix_3_perfam.csv", index_col=0) #pairwise alignment matrix aren shared with me
minimum_free_energy_table = pd.read_csv("mfes_3_perfam.csv", index_col=0) #minimum free energies provided by aren
ids = alignment_score_matrix.index.tolist() #get indices which are in row and column names 
fasta_df = pd.read_csv("fasta_df.csv") #contains sequence_id, sequence, and length for every sequence
fasta_df = fasta_df.iloc[ids] #only keep the ones from ids
X = torch.stack([representations[i] for i in ids]) #stack and get the required indices
rfam_ids = rfam_ids.iloc[ids] #only keep the ones from ids
minimum_free_energy_table = minimum_free_energy_table.reindex(ids) #reordering (because mfe order was different from asm order)

In [48]:
X_np = X.detach().cpu().numpy().astype(np.float32)
X_scaled = StandardScaler().fit_transform(X_np)
length = fasta_df["length"].to_numpy(dtype=float)
family = rfam_ids["rfam_id"].astype(str).to_numpy()

In [61]:
rng = np.random.default_rng(0) #use this seed for shufflings
def knn_stats(X, family, length, k, B=500):
    model = NearestNeighbors(n_neighbors=k+1, metric="cosine").fit(X) #k+1 because the first neighboor to a point is always itself
    distances, indices = model.kneighbors(X)
    neighbor_indices = indices[:, 1:] #exclude the points itself
    
    matches = family[neighbor_indices] == family[:, np.newaxis] #check if neighbor families match the point's family
    obs_purity = np.mean(matches)

    neighbor_lengths = length[neighbor_indices]
    obs_len_std = np.mean(np.std(neighbor_lengths, axis=1)) # Check variation in neighbor lengths

    family_nulls = np.zeros(B)
    length_nulls = np.zeros(B)

    for i in range(B):
        shuffled_fam = rng.permutation(family)
        shuffled_len = rng.permutation(length)
        
        shuf_matches = shuffled_fam[neighbor_indices] == shuffled_fam[:, np.newaxis]
        family_nulls[i] = np.mean(shuf_matches)
        
        shuf_neighbor_lens = shuffled_len[neighbor_indices]
        length_nulls[i] = np.mean(np.std(shuf_neighbor_lens, axis=1))
        
    fam_p = (1 + np.sum(family_nulls >= obs_purity)) / (B + 1)
    len_p = (1 + np.sum(length_nulls <= obs_len_std)) / (B + 1)
    return obs_purity, fam_p, obs_len_std, len_p
    
for k_val in [5, 10, 20, 50]:
    purity, p_fam, l_std, p_len = knn_stats(X_scaled, family, length, k_val)
    print("K val = " + str(k_val) + " | Purity (Family) = " + str(purity) + " (p = " + str(p_fam) + ")" + " | STD (Length) = " + str(l_std) + " (p = " + str(p_len) + ")") 

K val = 5 | Purity (Family) = 0.29434697855750486 (p = 0.001996007984031936) | STD (Length) = 69.69566062398422 (p = 0.001996007984031936)
K val = 10 | Purity (Family) = 0.1567251461988304 (p = 0.001996007984031936) | STD (Length) = 87.92971641274747 (p = 0.001996007984031936)
K val = 20 | Purity (Family) = 0.0827485380116959 (p = 0.001996007984031936) | STD (Length) = 102.86881714658465 (p = 0.001996007984031936)
K val = 50 | Purity (Family) = 0.0352046783625731 (p = 0.001996007984031936) | STD (Length) = 114.21109446447046 (p = 0.001996007984031936)


In [68]:
length_X = length.reshape(-1, 1) # shape (N, 1)
rng = np.random.default_rng(0) #use this seed for shufflings
def knn_stats(length_X, family, k, B=500):
    model = NearestNeighbors(n_neighbors=k+1, metric="minkowski").fit(length_X) #k+1 because the first neighboor to a point is always itself
    distances, indices = model.kneighbors(length_X)
    neighbor_indices = indices[:, 1:] #exclude the points itself
    
    matches = family[neighbor_indices] == family[:, np.newaxis] #check if neighbor families match the point's family
    obs_purity = np.mean(matches)

    family_nulls = np.zeros(B)

    for i in range(B):
        shuffled_fam = rng.permutation(family)
        
        shuf_matches = shuffled_fam[neighbor_indices] == shuffled_fam[:, np.newaxis]
        family_nulls[i] = np.mean(shuf_matches)
           
    fam_p = (1 + np.sum(family_nulls >= obs_purity)) / (B + 1)
    return obs_purity, fam_p,
    
for k_val in [5, 10, 20, 50]:
    purity, p_fam = knn_stats(length_X, family, k_val)
    print("K val = " + str(k_val) + " | Purity (Family) = " + str(purity) + " (p = " + str(p_fam) + ")")

K val = 5 | Purity (Family) = 0.1875243664717349 (p = 0.001996007984031936)
K val = 10 | Purity (Family) = 0.11150097465886939 (p = 0.001996007984031936)
K val = 20 | Purity (Family) = 0.06198830409356725 (p = 0.001996007984031936)
K val = 50 | Purity (Family) = 0.0304093567251462 (p = 0.001996007984031936)
